# Module 14: Training NER Models (Detailed)


## 🏷️ Training a Custom Named Entity Recognizer

While spaCy comes with robust pre-trained models, they only recognize generalized entities like `PERSON`, `ORG`, or `GPE` (Geopolitical Entity).
If you are working in a specialized domain—like parsing medical records, financial contracts, or technical logs—you need the model to recognize completely custom entities.

In this notebook, we will build a comprehensive pipeline to train a model to recognize `TECH_BRAND` and `DEVICE_MODEL` entities.


<br><br>

---

<br><br>


### 1. Generating Training Data (The Dictionary Format)

Before we can use the `DocBin` format, we usually start with raw Python dictionaries (or JSON files) containing our text and character offsets.

**Best Practice:** Always have a separate `TRAIN` dataset and a `DEV` (development/validation) dataset. Never evaluate your model on the data it was trained on!


In [4]:
import spacy
from spacy.tokens import DocBin
import warnings

# Suppress UserWarnings for clean output
warnings.filterwarnings("ignore")

# Raw Training Data
TRAIN_DATA = [
    ("I bought the new Apple iPhone 14 Pro Max yesterday.", {
        "entities": [(17, 22, "TECH_BRAND"), (23, 44, "DEVICE_MODEL")]
    }),
    ("The Samsung Galaxy S23 Ultra has a great camera.", {
        "entities": [(4, 11, "TECH_BRAND"), (12, 32, "DEVICE_MODEL")]
    }),
    ("I need to charge my Sony PlayStation 5.", {
        "entities": [(20, 24, "TECH_BRAND"), (25, 38, "DEVICE_MODEL")]
    }),
    ("Google released the Pixel 8 Pro last week.", {
        "entities": [(0, 6, "TECH_BRAND"), (20, 31, "DEVICE_MODEL")]
    }),
    ("I'm typing this on a Microsoft Surface Pro 9.", {
        "entities": [(21, 30, "TECH_BRAND"), (31, 44, "DEVICE_MODEL")]
    })
]

# Evaluation Data (Model has never seen these exact sentences)
DEV_DATA = [
    ("Is the Apple iPad Air worth buying?", {
        "entities": [(7, 12, "TECH_BRAND"), (13, 21, "DEVICE_MODEL")]
    }),
    ("My Samsung Galaxy Watch is broken.", {
        "entities": [(3, 10, "TECH_BRAND"), (11, 23, "DEVICE_MODEL")]
    })
]


<br><br>

---

<br><br>


### 2. Handling Token Boundary Errors

One of the most common errors in spaCy training is providing character offsets that don't align with token boundaries. For example, if you say an entity starts at character 5, but character 5 is in the middle of a word, `doc.char_span` will return `None`, crashing your script.

Let's write a robust converter function that handles misaligned tokens gracefully.


In [5]:
def robust_converter(data, output_file, model="en_core_web_sm"):
    nlp = spacy.blank("en") # Always use a blank model for tokenization to be safe
    db = DocBin()
    
    skipped_entities = 0
    
    for text, annot in data:
        doc = nlp.make_doc(text)
        ents = []
        for start, end, label in annot["entities"]:
            # alignment_mode="contract" ensures that if we are slightly off by a space,
            # spaCy will snap to the nearest valid token boundaries instead of returning None!
            span = doc.char_span(start, end, label=label, alignment_mode="contract")
            
            if span is None:
                print(f"[WARNING] Skipping entity [{start}:{end}] in text: '{text}'")
                skipped_entities += 1
            else:
                ents.append(span)
                
        try:
            doc.ents = ents
            db.add(doc)
        except ValueError as e:
            print(f"[ERROR] Overlapping entities in text: '{text}' - {e}")
            
    db.to_disk(output_file)
    print(f"Successfully saved to {output_file} (Skipped {skipped_entities} bad entities)")

# Convert the data
robust_converter(TRAIN_DATA, "train_ner.spacy")
robust_converter(DEV_DATA, "dev_ner.spacy")


Successfully saved to train_ner.spacy (Skipped 0 bad entities)
Successfully saved to dev_ner.spacy (Skipped 0 bad entities)


<br><br>

---

<br><br>


### 3. Generating the `config.cfg`

To train, we need to generate a `config.cfg`. We can do this programmatically via the CLI. 
We will use `init config` and specify `--pipeline ner` and `--optimize efficiency` to use the CNN (Tok2Vec) architecture.


In [6]:
import os

# Generate the base config using a shell command
print("Generating config.cfg...")
os.system("python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency --force")
print("config.cfg generated successfully!")

# Let's read the first 20 lines to see what it looks like
with open("config.cfg", "r") as f:
    print("".join(f.readlines()[:20]))


Generating config.cfg...
config.cfg generated successfully!
[paths]
train = null
dev = null
vectors = null
init_tok2vec = null

[system]
gpu_allocator = null
seed = 0

[nlp]
lang = "en"
pipeline = ["tok2vec", "ner"]
batch_size = 1000
disabled = []
before_creation = null
after_creation = null
after_pipeline_creation = null

[corpora]



<br><br>

---

<br><br>


### 4. Running the Training Loop (Mock/Demonstration)

In a real-world scenario, you run this in your terminal. Here is the exact command you use:

```bash
python -m spacy train config.cfg --output ./ner_models --paths.train ./train_ner.spacy --paths.dev ./dev_ner.spacy
```

When it runs, it outputs a table that looks like this:

```text
=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     32.50    0.00    0.00    0.00    0.00
 10     200         45.21   1421.10   45.20   42.10   50.00    0.45
 20     400         12.50    210.45   88.50   89.00   88.00    0.88
 30     600          2.10     45.20   95.20   96.00   94.50    0.95
```


<br><br>

---

<br><br>


### 5. Catastrophic Forgetting & How to Avoid It

If you want to train an existing model (like `en_core_web_sm`) rather than a blank one, you must be careful.

If you feed `en_core_web_sm` the 5 sentences above, it will optimize its weights *only* for `TECH_BRAND` and `DEVICE_MODEL`. The neural network will literally "forget" what a `PERSON` or `ORG` is because those labels weren't in the training data!

**To avoid Catastrophic Forgetting:**
1. **Use a Blank Model:** The safest approach. Train a separate model just for Tech, and use multiple models in your codebase.
2. **Pseudo-Rehearsal:** Mix old examples into your new data. You literally pass sentences containing `PERSON` and `ORG` labels alongside your new data so the network remembers them.
3. **Freezing Layers:** In the `config.cfg`, set `frozen_components = ["tok2vec"]`. This locks the base CNN, so you only train the topmost NER layer, preserving the model's core understanding of English grammar.


<br><br>

---

<br><br>


### 6. Loading and Testing the Trained Model

Once training finishes, spaCy saves two models in the `--output` folder:
- `model-best`: The model with the highest DEV SCORE (F1-score).
- `model-last`: The model from the final training epoch.

You almost always want to load `model-best`.


In [7]:
"""
# Once trained, you load it like any standard model!
import spacy

# Load the best model from disk
custom_nlp = spacy.load("./ner_models/model-best")

# Test it on unseen text
test_doc = custom_nlp("I am thinking about upgrading to the Apple iPhone 15 Pro.")

for ent in test_doc.ents:
    print(f"Text: {ent.text:<20} | Label: {ent.label_}")
"""


'\n# Once trained, you load it like any standard model!\nimport spacy\n\n# Load the best model from disk\ncustom_nlp = spacy.load("./ner_models/model-best")\n\n# Test it on unseen text\ntest_doc = custom_nlp("I am thinking about upgrading to the Apple iPhone 15 Pro.")\n\nfor ent in test_doc.ents:\n    print(f"Text: {ent.text:<20} | Label: {ent.label_}")\n'